# 🧬 GROMACS Precompiled CUDA Pipeline (With Live Progress Monitoring)
### Validation of Convergent Steric Gate Remodeling in Beetle Luciferases
**Author:** Bruno ("Blu") Stein Barbosa Menechino ([PMID: 38049805](https://pubmed.ncbi.nlm.nih.gov/38049805/))  
**Engine:** Precompiled GROMACS with native NVIDIA CUDA acceleration (`nompi_cuda`) via conda-forge.  
**Hardware:** NVIDIA T4 / V100 / A100 GPU (Google Colab Pro).  
**Real-Time Feedback:** Includes an active live monitor that prints percentage completion `[XX.X%]`, nanoseconds simulated, and GPU speed every 30 seconds!

---
### 🚀 How to Run:
1. Ensure your runtime is GPU: **Runtime -> Change runtime type -> T4 GPU**.
2. Run **Step 1** (Installs CondaColab; kernel automatically restarts).
3. Run **Step 2 through Step 6** sequentially.

## Step 1: Install CondaColab (Initializes Conda Environment)

In [ ]:
# Install condacolab helper
!pip install -q condacolab
import condacolab
condacolab.install()
# Note: Google Colab will display a prompt indicating the kernel has restarted. This is expected!

## Step 2: Install Precompiled GROMACS with CUDA (`nompi_cuda`)

In [ ]:
# Install precompiled GROMACS with CUDA support from conda-forge (takes ~90 seconds)
!mamba install -y -q -c conda-forge "gromacs=*=nompi_cuda*"
!pip install -q gmx_MMPBSA acpype py3Dmol matplotlib seaborn pandas

import os, sys, glob, base64, gzip
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Verify native CUDA support in GROMACS
print("==================================================================")
print("⚡ Verifying Precompiled CUDA GROMACS Installation:")
print("==================================================================")
!gmx --version | head -n 18

## Step 3: Unpack Embedded Holo Complexes into Workspace

In [ ]:
# Self-extracting embedded holo complexes into workspace
BASE_DIR = os.path.abspath("md_simulation")
os.makedirs(os.path.join(BASE_DIR, "inputs"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "outputs"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "mdp"), exist_ok=True)

embedded_payloads = {
  "1_A0A8K0DA02_TrueLuc_WT_S347_SLU_holo.pdb": "...",
  "2_A0A8K0C7P5_Paralog_WT_L347_SLU_holo.pdb": "...",
  "3_A0A8K0C7P5_Paralog_SAESteered_S347A348N349_SLU_holo.pdb": "...",
  "4_SLU_ligand_4G36_aligned.pdb": "H4sIAAAAAAAC/42XS27bMBCG9wZ8B15AxDxIDrlUZbU1akuB7Sxy/4t0SCqtA9EtvYghRPzyz+sf5jZfx9svYwya8329jI/5ZO6XdzOe5uUjP5rz8phv1/l0zg/fb+vVvJ2+GfeDw/Fw+zxN5v7+Nt/e1rue1xPrn5fMND7Gy8fjPJllOK3X8byYt3X6NT8UNl7n4+Hn/BgfV1M0rJxM+fuX8pg/A5N1AfQxWIdoBnSWWH+HFsAw5J9/P+szj8yEYc9jy0z6KBYRlMdWyL3gTc88Vp7b85z1MRWeQyn6nPgenjML+hZPwBUexByvt8KveMszz6s+3vO8DS7HGy0Ql3gjS4++YO5Ie17YzkeLWt+sD/NrTd79mSeqD1v6oucSLwspL2h9Yo++aKZoWvrQ1Xqg5nHQ7xCoh5fMPZmWPnBb/7mcv2gjQke8+tLk2vpk42keBwKLIXTo09xNvsmDJIUH2icDofXYkz8t2hQaPLFEVR9q3gaibV7+y2Oz5sTseSj5vPaxaD2IbZDUMb86axM39TGHoi8FjReTDdAzbzpri7TjDdUPSKTUl1/Ox/O8qbdM1OBF9avP+aVSXxe66pHno8nj6Eq8ofDQRuiqRzSLg5b/EdV5C+pbA2p9X9bjS7w6H6pxx0Ob8lgX/8t+ivqdOuZDm2zNjbr3e8y2mPvPZ38BKy509Avp/sDU0hc9Fr9ixsJjkB4emZUa+QMbePMrrPkj7NKn+4OwxWOu+iil4s8Ye/xAl9ZEe38m9afiVxov+sJzL/flF57uD+IWr+6zZL3Est8g9uwPCpo/afEguVx+62GL11FP/nQ+aL9/SfuuzEMqPpP1pZR69EXVt+8/yr4Hleeg7nPsuW9QUn2+pU9i9QMp8+Ytux6/Yp0PCu16hOpXXOOll7xnfdoEC/sWL+Qz2Q8Eyv71L/3+2Q/0pYmb/Ydlf4v1ZV+K7sue/tOhWhhb9UhY/dmDr/7sQ4++vD+gxZPg6/yW+5Xujy5/1twtlFr1jcDb/uDq96nnvpYvx9yYt6Scmj9X7ldaH9/j97q0JnYtHoR6H6Lkij7w0MPT/cGx5Vd1PqTcYzIvpa56JOVJ0/9c5bkktV+66quHJg6teJPf+pm3fk7/8Cv99+Z4mJfT8fAb1kdqQQ0NAAA="
}

for fname, b64_str in embedded_payloads.items():
    out_path = os.path.join(BASE_DIR, "inputs", fname)
    raw_data = gzip.decompress(base64.b64decode(b64_str))
    with open(out_path, "wb") as fh:
        fh.write(raw_data)
    print(f"  Extracted: {out_path} ({len(raw_data)} bytes)")

print(f"\n✅ All 4 holo structures unpacked successfully into {BASE_DIR}/inputs/.")
!ls -lh {BASE_DIR}/inputs/

## Step 4: Write Optimized GPU MDP Configuration Files (HMR dt = 4 fs)

In [ ]:
BASE_DIR = os.path.abspath("md_simulation")
mdp_dir = os.path.join(BASE_DIR, "mdp")
os.makedirs(mdp_dir, exist_ok=True)

# 1. Energy Minimization (em.mdp)
with open(os.path.join(mdp_dir, "em.mdp"), "w") as f:
    f.write('''integrator               = steep
emtol                    = 500.0
emstep                   = 0.01
nsteps                   = 5000
cutoff-scheme            = Verlet
nstlist                  = 20
rlist                    = 1.0
coulombtype              = PME
rcoulomb                 = 1.0
vdwtype                  = Cut-off
rvdw                     = 1.0
pbc                      = xyz
''')

# 2. NVT Equilibration (nvt.mdp, 100 ps, 300 K)
with open(os.path.join(mdp_dir, "nvt.mdp"), "w") as f:
    f.write('''define                   = -DPOSRES
integrator               = md
dt                       = 0.002
nsteps                   = 50000     ; 100 ps
cutoff-scheme            = Verlet
nstlist                  = 20
rlist                    = 1.0
coulombtype              = PME
rcoulomb                 = 1.0
vdwtype                  = Cut-off
rvdw                     = 1.0
tcoupl                   = V-rescale
tc-grps                  = System
tau_t                    = 1.0
ref_t                    = 300
pcoupl                   = no
pbc                      = xyz
gen_vel                  = yes
gen_temp                 = 300
gen_seed                 = -1
constraints              = all-bonds
continuation             = no
''')

# 3. NPT Equilibration (npt.mdp, 200 ps, 1 bar)
with open(os.path.join(mdp_dir, "npt.mdp"), "w") as f:
    f.write('''define                   = -DPOSRES
integrator               = md
dt                       = 0.002
nsteps                   = 100000    ; 200 ps
cutoff-scheme            = Verlet
nstlist                  = 20
rlist                    = 1.0
coulombtype              = PME
rcoulomb                 = 1.0
vdwtype                  = Cut-off
rvdw                     = 1.0
tcoupl                   = V-rescale
tc-grps                  = System
tau_t                    = 1.0
ref_t                    = 300
pcoupl                   = C-rescale
pcoupltype               = isotropic
tau_p                    = 2.0
ref_p                    = 1.0
compressibility          = 4.5e-5
pbc                      = xyz
constraints              = all-bonds
continuation             = yes
''')

# 4. Production MD (prod_hmr.mdp, dt = 4 fs, 10.0 ns = 2,500,000 steps on GPU)
with open(os.path.join(mdp_dir, "prod_hmr.mdp"), "w") as f:
    f.write('''integrator               = md
dt                       = 0.004     ; 4 fs step with HMR (-heavyh)
nsteps                   = 2500000   ; 10.0 ns trajectory (~140-180 ns/day on GPU)
nstxout-compressed       = 5000      ; 500 frames (every 20 ps)
nstenergy                = 5000
cutoff-scheme            = Verlet
nstlist                  = 40
rlist                    = 1.0
coulombtype              = PME
rcoulomb                 = 1.0
vdwtype                  = Cut-off
rvdw                     = 1.0
tcoupl                   = V-rescale
tc-grps                  = System
tau_t                    = 1.0
ref_t                    = 300
pcoupl                   = C-rescale
pcoupltype               = isotropic
tau_p                    = 5.0
ref_p                    = 1.0
compressibility          = 4.5e-5
pbc                      = xyz
constraints              = all-bonds
continuation             = yes
''')

print(f"✅ GPU MDP configuration files written in {mdp_dir}.")

## Step 5: Run Automated GPU Molecular Dynamics Loop with Live Feedback

In [ ]:
%%bash
set -e

BASE_DIR="$(pwd)/md_simulation"
INP_DIR="${BASE_DIR}/inputs"
MDP_DIR="${BASE_DIR}/mdp"
OUT_DIR="${BASE_DIR}/outputs"

COMPLEXES=(
  "1_A0A8K0DA02_TrueLuc_WT_S347_SLU_holo"
  "2_A0A8K0C7P5_Paralog_WT_L347_SLU_holo"
  "3_A0A8K0C7P5_Paralog_SAESteered_S347A348N349_SLU_holo"
)

for COMP in "${COMPLEXES[@]}"; do
    PDB_INPUT="${INP_DIR}/${COMP}.pdb"
    if [ ! -f "${PDB_INPUT}" ]; then
        echo "❌ Skipping ${COMP}: ${PDB_INPUT} not found!"
        continue
    fi
    echo "=================================================================="
    echo "🚀 Processing Complex on NVIDIA GPU: ${COMP}"
    echo "Input: ${PDB_INPUT}"
    echo "=================================================================="

    COMP_OUT="${OUT_DIR}/${COMP}"
    mkdir -p "${COMP_OUT}"
    cd "${COMP_OUT}"

    if [ ! -f "em.gro" ]; then
        echo "  [1/6] Extracting protein coordinates..."
        grep "^ATOM" "${PDB_INPUT}" > protein.pdb

        echo "  [2/6] Building AMBER99SB-ILDN topology with HMR (-heavyh)..."
        gmx pdb2gmx -f protein.pdb -o protein_hmr.gro -p topol.top -ff amber99sb-ildn -water tip3p -ignh -heavyh

        echo "  [3/6] Setting up Rhombic Dodecahedron box (1.0 nm buffer)..."
        gmx editconf -f protein_hmr.gro -o box.gro -c -d 1.0 -bt dodecahedron

        echo "  [4/6] Solvating with TIP3P water..."
        gmx solvate -cp box.gro -cs spc216.gro -o solv.gro -p topol.top

        echo "  [5/6] Adding 0.15 M NaCl ions for physiological neutralization..."
        gmx grompp -f "${MDP_DIR}/em.mdp" -c solv.gro -p topol.top -o ions.tpr -maxwarn 2
        echo "SOL" | gmx genion -s ions.tpr -o solv_ions.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15

        echo "  [6/6] Running steepest-descent energy minimization..."
        gmx grompp -f "${MDP_DIR}/em.mdp" -c solv_ions.gro -p topol.top -o em.tpr -maxwarn 2
        gmx mdrun -deffnm em -v
    else
        echo "  ✅ Energy minimization already completed (em.gro exists). Resuming from NVT!"
    fi

    if [ ! -f "nvt.gro" ]; then
        echo "  ⚡ Running NVT Equilibration on GPU (100 ps, 300 K)..."
        gmx grompp -f "${MDP_DIR}/nvt.mdp" -c em.gro -r em.gro -p topol.top -o nvt.tpr -maxwarn 2
        gmx mdrun -deffnm nvt -nb gpu -pme gpu
        echo "  ✅ NVT equilibration finished."
    else
        echo "  ✅ NVT equilibration already completed."
    fi

    if [ ! -f "npt.gro" ]; then
        echo "  ⚡ Running NPT Equilibration on GPU (200 ps, 1.0 bar)..."
        gmx grompp -f "${MDP_DIR}/npt.mdp" -c nvt.gro -r nvt.gro -p topol.top -o npt.tpr -maxwarn 2
        gmx mdrun -deffnm npt -nb gpu -pme gpu
        echo "  ✅ NPT equilibration finished."
    else
        echo "  ✅ NPT equilibration already completed."
    fi

    if [ ! -f "prod.xtc" ]; then
        echo "  ⚡ Launching Production MD on NVIDIA GPU (10.0 ns, dt = 4.0 fs HMR)..."
        gmx grompp -f "${MDP_DIR}/prod_hmr.mdp" -c npt.gro -t npt.cpt -p topol.top -o prod.tpr -maxwarn 2
        
        # Run mdrun in background and actively monitor progress every 30 seconds
        gmx mdrun -deffnm prod -nb gpu -pme gpu -bonded gpu -update gpu -v > prod_terminal.log 2>&1 &
        MDRUN_PID=$!
        TOTAL_STEPS=2500000

        echo "  --> Production running on GPU (PID: $MDRUN_PID). Streaming live progress updates every 30s..."
        echo "  ----------------------------------------------------------------"
        while kill -0 $MDRUN_PID 2>/dev/null; do
            sleep 30
            if [ -f "prod.log" ]; then
                CURR_STEP=$(grep "Step " prod.log | tail -n 1 | awk '{print $2}')
                if [ -n "$CURR_STEP" ] && [ "$CURR_STEP" -eq "$CURR_STEP" ] 2>/dev/null; then
                    PCT=$(awk -v c="$CURR_STEP" -v t="$TOTAL_STEPS" 'BEGIN { printf "%.1f", (c/t)*100 }')
                    NS=$(awk -v c="$CURR_STEP" 'BEGIN { printf "%.2f", c*0.004 }')
                    echo "  ⚡ [GROMACS GPU] [${PCT}%] ${NS} / 10.00 ns completed (Step ${CURR_STEP}/${TOTAL_STEPS})"
                else
                    echo "  ⏳ [GROMACS GPU] Simulation actively running on GPU..."
                fi
            fi
        done
        wait $MDRUN_PID
        echo "  ✅ Production MD finished on GPU for ${COMP}!"
    else
        echo "  ✅ Production MD already completed."
    fi

    echo "  --> Post-processing periodic boundary conditions (PBC)..."
    echo "1 0" | gmx trjconv -s prod.tpr -f prod.xtc -o prod_noPBC.xtc -pbc mol -center
    echo "4 4" | gmx rms -s prod.tpr -f prod_noPBC.xtc -o rmsd_backbone.xvg

    echo "✅ Finished GPU simulation for ${COMP}!"
done

echo "=================================================================="
echo "🎉 ALL 3 GPU SIMULATIONS COMPLETED SUCCESSFULLY!"
echo "=================================================================="


## Step 6: Backbone RMSD Stability Visualization & Comparison

In [ ]:
OUT_DIR = os.path.abspath("md_simulation/outputs")
complexes = [
    ("True Luc WT S347", "1_A0A8K0DA02_TrueLuc_WT_S347_SLU_holo", "#10b981"),
    ("Paralog WT L347", "2_A0A8K0C7P5_Paralog_WT_L347_SLU_holo", "#e11d48"),
    ("SAE-Steered Gate S347/A348/N349", "3_A0A8K0C7P5_Paralog_SAESteered_S347A348N349_SLU_holo", "#2563eb")
]

plt.figure(figsize=(10, 5), dpi=150)
for label, comp_dir, color in complexes:
    xvg_file = os.path.join(OUT_DIR, comp_dir, "rmsd_backbone.xvg")
    if os.path.exists(xvg_file):
        data = []
        with open(xvg_file, "r") as f:
            for line in f:
                if not line.startswith(("@", "#")):
                    parts = line.strip().split()
                    if len(parts) >= 2:
                        data.append([float(parts[0]) / 1000.0, float(parts[1]) * 10.0])  # Convert to ns and Angstroms
        if data:
            arr = np.array(data)
            plt.plot(arr[:, 0], arr[:, 1], label=f"{label} (Final={arr[-1, 1]:.2f} Å)", color=color, lw=1.8)

plt.xlabel("Simulation Time (ns)", fontweight="bold")
plt.ylabel("Backbone RMSD (Å)", fontweight="bold")
plt.title("GROMACS Precompiled CUDA MD: Pocket Backbone Stability (10 ns)", fontweight="bold")
plt.grid(alpha=0.3)
plt.legend(loc="lower right")
plt.savefig(os.path.join(OUT_DIR, "gromacs_rmsd_comparison.png"), dpi=200)
plt.show()
print("✅ Trajectory RMSD analysis plot generated.")